In [1]:
# Parameters
megadescriptor_version = 'S-224'  # 'S-224', 'B-224', 'L-384'
detection = '' # '', _detected', '_detected_manual'
seed = 42
query_ratio = 0.2

In [2]:
import torch
import numpy as np
import joblib
import torch.nn as nn
import torch.optim as optim
import random
from collections import defaultdict

In [10]:


def show_misclassified(y_true_tensor: torch.Tensor,
                       preds_tensor: torch.Tensor,
                       idx_array: np.ndarray,
                       detection_str: str):
    """Print misclassified examples using the index‑map CSV.

    Arguments:
        y_true_tensor: ground truth labels as torch tensor
        preds_tensor: predicted labels as torch tensor (same length)
        idx_array: numpy array of original indices corresponding to the
                   samples in y_true_tensor/preds_tensor (e.g. idx_test)
        detection_str: value of `detection` parameter used to pick dataset
    """
    import pandas as pd
    import os

    base_dir_root = r"C:\BP\pythonProject1\data_rysy"
    if detection_str == '':
        base_dir = os.path.join(base_dir_root, "rys_trening_data_Beno")
    elif detection_str == '_detected':
        base_dir = os.path.join(base_dir_root, "rys_trening_data_Beno_detected")
    elif detection_str == '_detected_manual':
        base_dir = os.path.join(base_dir_root, "rys_trening_data_Beno_detected_manual", "rys_trening_data_Beno_detected_manual")
    else:
        raise ValueError(f"unknown detection value: {detection_str}")

    csv_path = os.path.join(base_dir, "index_map.csv")
    index_map = pd.read_csv(csv_path)

    wrong = (preds_tensor != y_true_tensor).nonzero(as_tuple=True)[0]
    print(f"Number wrong: {len(wrong)}")
    for test_idx in wrong.tolist():
        true_label = encoder.inverse_transform([y_true_tensor[test_idx].item()])[0]
        pred_label = encoder.inverse_transform([preds_tensor[test_idx].item()])[0]
        original_idx = idx_array[test_idx]
        image_path = index_map.at[original_idx, 'path']
        print(f"Index {test_idx} (Orig {original_idx}): {image_path}")
        print(f"  True: {true_label}, Predicted: {pred_label}")
        print()


In [3]:
np.random.seed(seed)

# Path to new file
data_path = f"saved_models/{megadescriptor_version}/data{detection}.npz"
encoder_path = f"saved_models/{megadescriptor_version}/label_encoder{detection}.pkl"

# Load everything at once
data = np.load(data_path)

embeddings = data["embeddings"]      # shape (N, D)
labels = data["label_ids"]           # integer labels
# original_labels = data["labels"]     # string labels (optional)

print("Embeddings shape:", embeddings.shape)

# Optional: load encoder if you want inverse_transform
encoder = joblib.load(encoder_path)
names = encoder.inverse_transform(labels)

encoder = joblib.load(encoder_path)
id_to_name = dict(enumerate(encoder.classes_))
name_to_id = {v: k for k, v in id_to_name.items()}


Embeddings shape: (319, 768)


In [4]:
from sklearn.model_selection import train_test_split

# Create an array of original indices to track which embedding each sample came from
original_indices = np.arange(len(embeddings))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    embeddings, labels, original_indices, test_size=query_ratio, random_state=seed
)

In [5]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# convert numpy arrays from the train/test split into torch tensors
# use float32 for the embeddings and long for the integer labels
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

# optional: keep full dataset tensors for evaluation later
X = torch.tensor(embeddings, dtype=torch.float32)
y = torch.tensor(labels, dtype=torch.long)

# build training dataset and loader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


In [6]:
class Classifier(nn.Module):
    def __init__(self, input_dim=768, num_classes=10, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [7]:
# determine number of classes from training labels
num_classes = len(torch.unique(torch.tensor(y_train)))

model = Classifier(input_dim=768, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    model.train()
    total_loss = 0
    correct = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == y_batch).sum().item()

    acc = correct / len(train_dataset)
    print(f"Epoch {epoch}: loss={total_loss:.3f}, acc={acc:.3f}")


Epoch 0: loss=40.272, acc=0.176
Epoch 1: loss=33.105, acc=0.337
Epoch 2: loss=25.523, acc=0.553
Epoch 3: loss=19.438, acc=0.620
Epoch 4: loss=14.450, acc=0.749
Epoch 5: loss=9.968, acc=0.847
Epoch 6: loss=7.381, acc=0.871
Epoch 7: loss=4.838, acc=0.941
Epoch 8: loss=3.088, acc=0.969
Epoch 9: loss=2.351, acc=0.980
Epoch 10: loss=1.877, acc=0.984
Epoch 11: loss=1.776, acc=0.973
Epoch 12: loss=1.332, acc=0.988
Epoch 13: loss=1.094, acc=0.992
Epoch 14: loss=0.887, acc=0.988
Epoch 15: loss=0.673, acc=0.996
Epoch 16: loss=0.624, acc=1.000
Epoch 17: loss=0.411, acc=1.000
Epoch 18: loss=0.410, acc=0.996
Epoch 19: loss=0.432, acc=0.996
Epoch 20: loss=0.366, acc=1.000
Epoch 21: loss=0.321, acc=0.996
Epoch 22: loss=0.323, acc=0.996
Epoch 23: loss=0.249, acc=1.000
Epoch 24: loss=0.212, acc=1.000
Epoch 25: loss=0.306, acc=0.992
Epoch 26: loss=0.259, acc=0.996
Epoch 27: loss=0.184, acc=1.000
Epoch 28: loss=0.246, acc=0.996
Epoch 29: loss=0.229, acc=1.000
Epoch 30: loss=0.331, acc=0.992
Epoch 31: los

In [8]:
model.eval()
with torch.no_grad():
    preds = model(X).argmax(1)
accuracy = (preds == y).float().mean()
print("Final train accuracy:", accuracy.item())


Final train accuracy: 0.9341692924499512


In [13]:
# reuse helper function defined earlier to list misclassified samples
show_misclassified(y_test_tensor, preds, idx_test, detection)

Number wrong: 19
Index 7 (Orig 186): rys_trening_data_Beno\Izidor\Izidor_27.JPG
  True: Izidor, Predicted: Milos

Index 12 (Orig 164): rys_trening_data_Beno\Eliska\Eliska_7.JPG
  True: Eliska, Predicted: Brano

Index 14 (Orig 211): rys_trening_data_Beno\Kiara\Kiara_17.JPG
  True: Kiara, Predicted: Benadik

Index 16 (Orig 185): rys_trening_data_Beno\Izidor\Izidor_26.JPG
  True: Izidor, Predicted: Milos

Index 18 (Orig 314): rys_trening_data_Beno\Zora\Zora_5.JPG
  True: Zora, Predicted: Izidor

Index 22 (Orig 197): rys_trening_data_Beno\Izidor\Izidor_37.JPG
  True: Izidor, Predicted: Albin

Index 23 (Orig 108): rys_trening_data_Beno\Brano\Brano_2.JPG
  True: Brano, Predicted: Albin

Index 25 (Orig 118): rys_trening_data_Beno\Dio\Dio_13.jpg
  True: Dio, Predicted: Roman

Index 26 (Orig 296): rys_trening_data_Beno\Roman\Roman_30.JPG
  True: Roman, Predicted: Benadik

Index 27 (Orig 33): rys_trening_data_Beno\Albin\Albin_36.JPG
  True: Albin, Predicted: Edo

Index 31 (Orig 5): rys_trening_d

In [ ]:
from proportional_split_xy import proportional_split_xy

## Poznamenanie k výsledkom tréningu

- **Izidor_27** (nočná fotka zozadu) bol nesprávne klasifikovaný ako **Miloš**, ktorý má v datasete veľa obrázkov zozadu, ale aj veľa nočných.
- **Eliška_7** sa pravdepodobne podobá na **Braňa**.
- **Kiara_17** je nočný dobre osvetlený záber zboku s kontrastným zatmeným pozadím, veľmi podobný mnohým zaberom **Romana** s týmito charakteristikami.
- **Izidor_26** je záber zboku s výnimočne zeleným pozadím, nesprávne klasifikovaný ako **Roman**, ktorý má v datasete (v porovnaní s ostatnými) výrazne veľa snímok zboku.
- **Zora_5** bola pre kombináciu sneh + ihličnany klasifikovaná ako **Izidor**, ktorý má v tréningovom sete veľa obrázkov tohto typu.
- **Izidor_37** (nočná fotka + svietiace oči) bol klasifikovaný ako **Miloš**, ktorý má v datasete veľa obrázkov s touto kombináciou.
- **Brano_2** (jesenná fotka) bol nesprávne klasifikovaný ako **Eliška**, u ktorej sú niektoré jesenné obrázky.

## CrossEntropy Loss


In [12]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# === Load data ===
X = torch.from_numpy(embeddings).float()
y = torch.from_numpy(labels).long()   # must correspond to embeddings order

print(f"Loaded: X={X.shape}, y={y.shape}")

# === Train/Val split ===
# X_train, X_test, y_train, y_test = proportional_split_xy(X, y, query_ratio=0.2)
# make sure the split arrays are tensors for TensorDataset
if not isinstance(X_train, torch.Tensor):
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
else:
    X_train_tensor = X_train

if not isinstance(y_train, torch.Tensor):
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
else:
    y_train_tensor = y_train

if not isinstance(X_test, torch.Tensor):
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
else:
    X_test_tensor = X_test

if not isinstance(y_test, torch.Tensor):
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)
else:
    y_test_tensor = y_test

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
val_ds   = TensorDataset(X_test_tensor, y_test_tensor)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=16, shuffle=False)

# === Compute class weights to handle imbalance ===
classes = torch.unique(y).numpy()
weights = compute_class_weight('balanced', classes=classes, y=y.numpy())
class_weights = torch.tensor(weights, dtype=torch.float)
print("Class weights:", class_weights)

# === Define model ===
class Classifier(nn.Module):
    def __init__(self, input_dim=768, num_classes=len(classes), hidden_dim=512, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = Classifier()
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# === Training loop ===
best_val_acc = 0
patience, patience_counter = 20, 0

for epoch in range(100):
    model.train()
    total_loss, correct = 0, 0
    for xb, yb in train_dl:
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
        correct += (out.argmax(1) == yb).sum().item()

    train_acc = correct / len(train_ds)

    # --- Validation ---
    model.eval()
    val_correct, val_loss = 0, 0
    val_preds_list = []
    val_idxs = []
    with torch.no_grad():
        for xb, yb in val_dl:
            out = model(xb)
            loss = criterion(out, yb)
            val_loss += loss.item() * len(xb)
            val_correct += (out.argmax(1) == yb).sum().item()
            val_preds_list.append(out.argmax(1))
    
    val_acc = val_correct / len(val_ds)
    print(f"Epoch {epoch:03d}: train_loss={total_loss/len(train_ds):.4f}, "
          f"train_acc={train_acc:.3f}, val_acc={val_acc:.3f}")

    # --- Early stopping ---
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

# === Evaluation on best model ===
model.load_state_dict(torch.load("best_model.pt"))
model.eval()
with torch.no_grad():
    preds = model(X_test_tensor).argmax(1)
val_acc = (preds == y_test_tensor).float().mean().item()
print(f"\n✅ Final validation accuracy: {val_acc:.3f}")

# print misclassified for validation/test
show_misclassified(y_test_tensor, preds, idx_test, detection)


Loaded: X=torch.Size([319, 768]), y=torch.Size([319])
Class weights: tensor([4.5571, 0.5996, 0.3560, 2.8482, 1.8988, 0.7857, 2.0714, 0.6158, 1.5190,
        2.5317, 0.4747, 0.8439, 3.2551, 2.5317])
Epoch 000: train_loss=2.5798, train_acc=0.180, val_acc=0.266
Epoch 001: train_loss=1.9178, train_acc=0.580, val_acc=0.422
Epoch 002: train_loss=1.1041, train_acc=0.675, val_acc=0.562
Epoch 003: train_loss=0.5802, train_acc=0.800, val_acc=0.547
Epoch 004: train_loss=0.2963, train_acc=0.914, val_acc=0.578
Epoch 005: train_loss=0.1524, train_acc=0.953, val_acc=0.625
Epoch 006: train_loss=0.0917, train_acc=0.980, val_acc=0.656
Epoch 007: train_loss=0.0570, train_acc=0.992, val_acc=0.641
Epoch 008: train_loss=0.0307, train_acc=0.992, val_acc=0.703
Epoch 009: train_loss=0.0208, train_acc=1.000, val_acc=0.672
Epoch 010: train_loss=0.0154, train_acc=1.000, val_acc=0.672
Epoch 011: train_loss=0.0138, train_acc=0.992, val_acc=0.672
Epoch 012: train_loss=0.0080, train_acc=1.000, val_acc=0.688
Epoch 013